In [2]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Update this path to where your CARE_MDS repository is located on Drive
PROJECT_ROOT = '/content/drive/MyDrive/CARE_MDS'
os.chdir(PROJECT_ROOT)
print(f"Current Working Directory: {os.getcwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Current Working Directory: /content/drive/MyDrive/CARE_MDS


In [3]:
!pip install -r requirements.txt
!python preprocessing/analyze_dataset.py

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 74.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 12.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.2/63.2 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 101.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.7/51.7 kB 6.1 MB/s eta 0:00:00
   

Traceback (most recent call last):
  File "/content/drive/MyDrive/CARE_MDS/preprocessing/analyze_dataset.py", line 2, in <module>
^C


In [22]:
!python /content/drive/MyDrive/CARE_MDS/preprocessing/analyze_dataset.py

✅ Successfully loaded Multi-News dataset from /content/drive/MyDrive/CARE_MDS/datasets/multi_news_saved
✅ Successfully loaded CNN/Daily-News dataset from /content/drive/MyDrive/CARE_MDS/datasets/cnn_dailymail_saved
Cache directory ready: cache/multinews
Cache directory ready: cache/cnn_dailynews

==================== Analyzing Dataset: Multi-News ====================

--- Processing Split: train ---
🔄 Loading tokenizer: allenai/led-large-16384...
✅ Tokenizer 'allenai/led-large-16384' is ready.
Analyzing 4000 documents...
Finished tokenization in 26.93s
🔄 Loading embedding model: all-MiniLM-L6-v2...
Loading weights: 100% 103/103 [00:00<00:00, 2137.04it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you ex

In [23]:
import torch
import time
import threading

def keep_gpu_alive(interval=60):
    """
    Runs a tiny GPU operation every 'interval' seconds to prevent timeout.
    """
    if not torch.cuda.is_available():
        print("GPU not available. Switching to CPU (won't prevent GPU timeout).")
        return

    print("Starting GPU keep-alive loop...")
    device = torch.device("cuda")
    # Create a small dummy tensor on GPU
    dummy = torch.rand(10, 10).to(device)

    while True:
        try:
            # Perform a tiny computation to reset the idle timer
            _ = dummy @ dummy.t()
            torch.cuda.synchronize()
            time.sleep(interval)
        except KeyboardInterrupt:
            print("\nStopped by user.")
            break



In [24]:
for i in range(0,50):
  # Start the loop in a background thread so you can still use the notebook
  threading.Thread(target=keep_gpu_alive, daemon=True).start()
  print("Keep-alive thread started. You can now continue your work.")

Starting GPU keep-alive loop...
Keep-alive thread started. You can now continue your work.
Starting GPU keep-alive loop...
Keep-alive thread started. You can now continue your work.
Starting GPU keep-alive loop...
Keep-alive thread started. You can now continue your work.
Starting GPU keep-alive loop...
Keep-alive thread started. You can now continue your work.
Starting GPU keep-alive loop...
Keep-alive thread started. You can now continue your work.
Starting GPU keep-alive loop...
Keep-alive thread started. You can now continue your work.
Starting GPU keep-alive loop...
Keep-alive thread started. You can now continue your work.
Starting GPU keep-alive loop...
Keep-alive thread started. You can now continue your work.
Starting GPU keep-alive loop...
Keep-alive thread started. You can now continue your work.
Starting GPU keep-alive loop...
Keep-alive thread started. You can now continue your work.
Starting GPU keep-alive loop...
Keep-alive thread started. You can now continue your work.

In [21]:
from transformers import AutoModelForSeq2SeqLM
from utils import get_tokenizer
try:
  print(f"📦 Persisting model checkpoint for {dataset_name}...")
  # Load the tokenizer and model
  tokenizer = get_tokenizer("allenai/led-large-16384")
  model = AutoModelForSeq2SeqLM.from_pretrained("allenai/led-large-16384")

  # Define output directory
  model_output_dir = f"models/{dataset_name.lower().replace('/', '_')}"

  # Use our utility to save the checkpoint
  save_fine_tuned_transformer(model, tokenizer, output_dir=model_output_dir)
except Exception as e:
  print(f"⚠️ Model persistence failed for {dataset_name}: {e}")

ModuleNotFoundError: No module named 'utils'